## Fetch trades and quotes for stock and one option

In [ ]:
if False:
    import os
    os.environ["QUANTDATA_CACHE_MODE"] = "off" # turns off global cache mode ... good for testing but not in general

%load_ext autoreload
%autoreload 1

%aimport openqai.core
%aimport openqai.core.biz_calendar
%aimport openqai.data.massive.client
%aimport openqai.data.massive.options.contracts
%aimport openqai.data.massive.processing

from datetime import date, datetime
import warnings
from openqai.data.massive.client import MassiveClient
from cdxcore.util import fmt_date, fmt_dict, to_date
from cdxcore.verbose import Context

import polars as pl

def test():

    ticker = "SPY"
    d      = to_date("2024-06-17")
    client = MassiveClient()
    option_ticker = "O:SPY240624C00462000"

    def package( ticker, verbose = Context.all ):

        with verbose.write_t(f"Fetching OCV, trades, quotes for {ticker} on {fmt_date(d)}... ", end='') as tme:
            ocv = client.fetch_day_summary(ticker, d, adjusted=True, override_cache_mode="on" )
            verbose.write(f"done: found {fmt_dict(ocv)}. This took {tme}.", head=False)

        with verbose.write_t(f"Fetching trades for {ticker} on {fmt_date(d)}... ", end='') as tme:
            trades = client.fetch_trades(ticker, d, override_cache_mode="on") # extracts trades, removed zero size and price, and excludes trades with NORMAL_TRADE_EXCLUDE_FLAGS
            verbose.write(f"done: found {len(trades)} trades. This took {tme}.", head=False)

        with verbose.write_t(f"Fetching quotes for {ticker} on {fmt_date(d)}... ", end='') as tme:
            quotes = client.fetch_quotes(ticker, d, override_cache_mode="on")
            verbose.write(f"done: found {len(quotes)} quotes. This took {tme}.", head=False)

        return ocv, trades, quotes  

    package(ticker)
    ocv, trades, quotes = package(option_ticker)

    display(ocv)
    display(trades.head())
    display(quotes.head())

test()



In [ ]:
d      = to_date("2024-06-17")
client = MassiveClient()
option_ticker = "O:SPY240624C00524000"

client.has_any_quotes_or_trades(option_ticker, d)



